# 📄 Medical Report Image Enhancement: Deblurring & Denoising
## Optimized for Google Colab Free Tier | ~336 Training Images | 150 Epochs

### Your Dataset Structure:
```
Medical_Report/
├── train/
│   ├── input/     ← your blurry/noisy medical report images
│   └── target/    ← matching clean ground truth images
└── val/
    ├── input/     ← validation blurry/noisy images
    └── target/    ← validation clean ground truth images
```

### Quick Start:
1. Upload dataset to Google Drive at: `MyDrive/Medical_Report/`
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells top to bottom (Runtime → Run all)
4. Training takes ~30–45 min on T4 GPU
5. Results saved automatically to Google Drive

### What this notebook does:
- **Classical Methods** (instant, no training): CLAHE, Non-local means, Unsharp mask
- **Deep Learning**: Lightweight U-Net trained on 50,000+ patches extracted from your 336 images
- **Resume support**: If Colab disconnects, re-run and it picks up from the best saved checkpoint
- **Full-image inference**: Works on any resolution via patch stitching

## Cell 1 · Environment Setup & GPU Detection

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted")
except Exception as e:
    print(f"⚠️  Drive not mounted: {e}")
    print("💡 Fix: Runtime → Change runtime type or re-run this cell")

In [ ]:
# ── GPU / System Detection ────────────────────────────────────────────────────
import subprocess, sys, os

HAS_GPU = False
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                        '--format=csv,noheader'], encoding='utf-8').strip()
    print(f"✅ GPU detected: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("⚠️  No GPU detected — running on CPU (training will be slow)")
    print("💡 Fix: Runtime → Change runtime type → T4 GPU")

# System RAM
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"💾 System RAM: {ram_gb:.1f} GB")
except ImportError:
    pass

In [ ]:
# ── Install / Import Dependencies ─────────────────────────────────────────────
try:
    import tensorflow as tf
    import numpy as np
    import cv2
    import matplotlib.pyplot as plt
    import random, gc, os, shutil
    from pathlib import Path
    print(f"✅ TensorFlow {tf.__version__} | NumPy {np.__version__} | OpenCV {cv2.__version__}")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("💡 Fix: Run: !pip install tensorflow opencv-python-headless matplotlib")

In [ ]:
# ── Reproducibility Seeds ─────────────────────────────────────────────────────
import random, os
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import numpy as np; np.random.seed(SEED)
    import tensorflow as tf; tf.random.set_seed(SEED)
    print("✅ Seeds set for reproducibility")
except Exception as e:
    print(f"⚠️  Seed warning: {e}")

In [ ]:
# ── Mixed Precision (halves VRAM on T4) ───────────────────────────────────────
try:
    import tensorflow as tf
    if HAS_GPU:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("✅ Mixed precision (float16) enabled — halves VRAM usage")
    else:
        tf.keras.mixed_precision.set_global_policy('float32')
        print("ℹ️  Running float32 (CPU mode)")
except Exception as e:
    print(f"⚠️  Mixed precision not set: {e}")

## Cell 2 · Configuration

> **Edit only this cell** to adjust hyperparameters or paths.

In [ ]:
# ===== USER CONFIGURATION =====
DATASET_PATH   = '/content/drive/MyDrive/Medical_Report/'

TRAIN_INPUT    = DATASET_PATH + 'train/input/'
TRAIN_TARGET   = DATASET_PATH + 'train/target/'
VAL_INPUT      = DATASET_PATH + 'val/input/'
VAL_TARGET     = DATASET_PATH + 'val/target/'

CHECKPOINT_DIR = '/content/drive/MyDrive/Medical_Report_checkpoints/'
RESULTS_DIR    = '/content/drive/MyDrive/Medical_Report_results/'

# Local (fast I/O) patch storage — NOT saved to Drive
PATCH_DIR      = '/content/patches/'

PATCH_SIZE     = 128      # 128×128 patches (saves 4× memory vs 256×256)
PATCH_STRIDE   = 64       # 50% overlap → more patches from each image
BATCH_SIZE     = 8        # Safe for T4 15 GB VRAM; try 16 if no OOM
EPOCHS         = 150      # EarlyStopping will stop earlier if converged
LEARNING_RATE  = 1e-3
IMG_CHANNELS   = 1        # Grayscale — medical reports are text documents

SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
# ===============================
print("✅ Configuration loaded")
print(f"   Patch size:  {PATCH_SIZE}×{PATCH_SIZE}  |  Stride: {PATCH_STRIDE}")
print(f"   Batch size:  {BATCH_SIZE}  |  Epochs: {EPOCHS}")
print(f"   Dataset:     {DATASET_PATH}")

## Cell 3 · Dataset Loading & Validation

In [ ]:
# ── Load & match file pairs ───────────────────────────────────────────────────
try:
    from pathlib import Path

    def list_images(folder):
        p = Path(folder)
        if not p.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")
        files = sorted([f for f in p.iterdir() if f.suffix.lower() in SUPPORTED_EXTS])
        return files

    train_inputs  = list_images(TRAIN_INPUT)
    train_targets = list_images(TRAIN_TARGET)
    val_inputs    = list_images(VAL_INPUT)
    val_targets   = list_images(VAL_TARGET)

    # Match by filename stem
    def match_pairs(inputs, targets):
        in_stems  = {f.stem: f for f in inputs}
        tgt_stems = {f.stem: f for f in targets}
        common    = sorted(set(in_stems) & set(tgt_stems))
        if not common:
            raise ValueError("No matching filenames between input/ and target/!")
        unmatched_in  = set(in_stems) - set(common)
        unmatched_tgt = set(tgt_stems) - set(common)
        if unmatched_in:
            print(f"  ⚠️  {len(unmatched_in)} input files have no matching target (skipped)")
        if unmatched_tgt:
            print(f"  ⚠️  {len(unmatched_tgt)} target files have no matching input (skipped)")
        return [(in_stems[s], tgt_stems[s]) for s in common]

    TRAIN_PAIRS = match_pairs(train_inputs, train_targets)
    VAL_PAIRS   = match_pairs(val_inputs,   val_targets)

    print(f"✅ Training pairs:   {len(TRAIN_PAIRS)}")
    print(f"✅ Validation pairs: {len(VAL_PAIRS)}")

    if len(TRAIN_PAIRS) == 0:
        raise ValueError("No training pairs found!")
    if len(VAL_PAIRS) == 0:
        raise ValueError("No validation pairs found!")

except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Fix: Ensure your Drive is mounted and folder structure matches:")
    print("   Medical_Report/train/input/ + Medical_Report/train/target/")
    print("   Medical_Report/val/input/   + Medical_Report/val/target/")
    raise

In [ ]:
# ── Preview 6 sample pairs ────────────────────────────────────────────────────
try:
    import cv2, matplotlib.pyplot as plt, numpy as np

    n_preview = min(6, len(TRAIN_PAIRS))
    fig, axes = plt.subplots(2, n_preview, figsize=(3 * n_preview, 6))
    fig.suptitle("Sample Training Pairs  |  Top: Input (degraded)  ·  Bottom: Target (clean)",
                 fontsize=12)

    for i in range(n_preview):
        inp_path, tgt_path = TRAIN_PAIRS[i]
        inp = cv2.imread(str(inp_path), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(str(tgt_path), cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(inp, cmap='gray'); axes[0, i].set_title(inp_path.name[:20], fontsize=8)
        axes[1, i].imshow(tgt, cmap='gray'); axes[1, i].set_title(tgt_path.name[:20], fontsize=8)
        axes[0, i].axis('off'); axes[1, i].axis('off')

    plt.tight_layout(); plt.show()
    print("✅ Sample pairs displayed")

except Exception as e:
    print(f"❌ Preview error: {e}")
    print("💡 Fix: Check that image files are valid (not corrupt)")

## Cell 4 · Patch Extraction

> **Why patches?** Instead of training on ~336 full images we extract
> 128×128 patches with 50% overlap → ~50,000 training patches.
> This is the **#1 fix** for small medical imaging datasets.

In [ ]:
# ── Patch extraction helper ───────────────────────────────────────────────────
import numpy as np, cv2, os
from pathlib import Path

def extract_patches(img, patch_size, stride):
    """Return list of (patch, row, col) tuples from a grayscale image."""
    patches = []
    h, w = img.shape[:2]
    for r in range(0, h - patch_size + 1, stride):
        for c in range(0, w - patch_size + 1, stride):
            patches.append(img[r:r+patch_size, c:c+patch_size])
    return patches

print("✅ Patch extraction helper defined")

In [ ]:
# ── Extract and save patches ─────────────────────────────────────────────────
try:
    import gc

    for split, pairs in [('train', TRAIN_PAIRS), ('val', VAL_PAIRS)]:
        inp_dir = Path(PATCH_DIR) / split / 'input'
        tgt_dir = Path(PATCH_DIR) / split / 'target'
        inp_dir.mkdir(parents=True, exist_ok=True)
        tgt_dir.mkdir(parents=True, exist_ok=True)

        count = 0
        for img_idx, (inp_path, tgt_path) in enumerate(pairs):
            inp = cv2.imread(str(inp_path), cv2.IMREAD_GRAYSCALE)
            tgt = cv2.imread(str(tgt_path), cv2.IMREAD_GRAYSCALE)

            if inp is None or tgt is None:
                print(f"  ⚠️  Could not read {inp_path.name} — skipped")
                continue

            # Resize target to match input if sizes differ
            if inp.shape != tgt.shape:
                tgt = cv2.resize(tgt, (inp.shape[1], inp.shape[0]),
                                 interpolation=cv2.INTER_LINEAR)

            inp_patches = extract_patches(inp, PATCH_SIZE, PATCH_STRIDE)
            tgt_patches = extract_patches(tgt, PATCH_SIZE, PATCH_STRIDE)

            for j, (ip, tp) in enumerate(zip(inp_patches, tgt_patches)):
                fname = f"{img_idx:04d}_{j:05d}.png"
                cv2.imwrite(str(inp_dir / fname), ip)
                cv2.imwrite(str(tgt_dir / fname), tp)
                count += 1

        print(f"✅ {split.capitalize()} patches saved: {count}")

    gc.collect()
    print("✅ Patch extraction complete")

except Exception as e:
    print(f"❌ Error during patch extraction: {e}")
    print("💡 Fix: Check that images can be read by OpenCV (valid format/path)")
    raise

In [ ]:
# ── Count and preview patches ─────────────────────────────────────────────────
try:
    from pathlib import Path
    import cv2, matplotlib.pyplot as plt, numpy as np, random

    n_train = len(list((Path(PATCH_DIR) / 'train' / 'input').glob('*.png')))
    n_val   = len(list((Path(PATCH_DIR) / 'val'   / 'input').glob('*.png')))
    print(f"📊 Training patches:   {n_train:,}")
    print(f"📊 Validation patches: {n_val:,}")

    # Show 8 sample patches (input | target)
    sample_files = random.sample(
        list((Path(PATCH_DIR) / 'train' / 'input').glob('*.png')), min(8, n_train)
    )
    fig, axes = plt.subplots(2, len(sample_files), figsize=(2 * len(sample_files), 4))
    fig.suptitle("Sample Patches  |  Top: Input · Bottom: Target", fontsize=10)
    for i, f in enumerate(sample_files):
        inp_p = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        tgt_p = cv2.imread(str(Path(PATCH_DIR) / 'train' / 'target' / f.name),
                           cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(inp_p, cmap='gray'); axes[0, i].axis('off')
        axes[1, i].imshow(tgt_p, cmap='gray'); axes[1, i].axis('off')
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f"❌ Preview error: {e}")
    print("💡 Fix: Re-run the patch extraction cell above")

## Cell 5 · Classical Enhancement Methods

> These methods require **zero training** and always produce results.
> Run this section to get instant baseline results even before deep learning.

In [ ]:
# ── Classical method implementations ─────────────────────────────────────────
import cv2, numpy as np

def compute_psnr(img1, img2):
    """PSNR between two uint8 grayscale images."""
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse  = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100.0
    return 10 * np.log10(255.0 ** 2 / mse)

def compute_ssim(img1, img2):
    """Simple SSIM approximation (no scikit-image dependency)."""
    try:
        from skimage.metrics import structural_similarity as ssim
        return ssim(img1, img2, data_range=255)
    except ImportError:
        # fallback: correlation-based approximation
        mu1, mu2   = img1.mean(), img2.mean()
        s1, s2     = img1.std(), img2.std()
        cov        = np.mean((img1 - mu1) * (img2 - mu2))
        C1, C2     = 6.5025, 58.5225
        return ((2*mu1*mu2 + C1) * (2*cov + C2)) /                ((mu1**2 + mu2**2 + C1) * (s1**2 + s2**2 + C2))

def clahe_enhance(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def nlm_denoise(img):
    return cv2.fastNlMeansDenoising(img, h=10, templateWindowSize=7,
                                    searchWindowSize=21)

def unsharp_mask(img, sigma=1.0, strength=1.5):
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    return cv2.addWeighted(img, 1 + strength, blurred, -strength, 0)

def adaptive_threshold(img):
    return cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 11, 2)

def combined_pipeline(img):
    """Denoise → Sharpen → CLAHE — best for medical text documents."""
    out = nlm_denoise(img)
    out = unsharp_mask(out)
    out = clahe_enhance(out)
    return out

print("✅ Classical methods defined")

In [ ]:
# ── Apply to validation images & compute metrics ─────────────────────────────
try:
    classical_results = {}  # method_name → {psnr: [], ssim: []}
    methods = {
        'CLAHE':          clahe_enhance,
        'NLM Denoise':    nlm_denoise,
        'Unsharp Mask':   unsharp_mask,
        'Adaptive Thresh':adaptive_threshold,
        'Combined':       combined_pipeline,
    }

    n_eval = min(10, len(VAL_PAIRS))
    for name in methods:
        classical_results[name] = {'psnr': [], 'ssim': []}

    for inp_path, tgt_path in VAL_PAIRS[:n_eval]:
        inp = cv2.imread(str(inp_path), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(str(tgt_path), cv2.IMREAD_GRAYSCALE)
        if inp is None or tgt is None:
            continue
        if inp.shape != tgt.shape:
            tgt = cv2.resize(tgt, (inp.shape[1], inp.shape[0]))
        for name, fn in methods.items():
            out = fn(inp)
            if out.shape != tgt.shape:
                out = cv2.resize(out, (tgt.shape[1], tgt.shape[0]))
            classical_results[name]['psnr'].append(compute_psnr(out, tgt))
            classical_results[name]['ssim'].append(compute_ssim(out, tgt))

    print("\n📊 Classical Methods Results (avg over first 10 val images):")
    print(f"  {'Method':<20} {'PSNR (dB)':>10} {'SSIM':>8}")
    print("  " + "-" * 40)
    for name, vals in classical_results.items():
        if vals['psnr']:
            print(f"  {name:<20} {np.mean(vals['psnr']):>10.2f} {np.mean(vals['ssim']):>8.4f}")

except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Fix: Ensure validation images are readable")

In [ ]:
# ── Visual comparison (one image, all methods) ───────────────────────────────
try:
    import matplotlib.pyplot as plt
    inp_path, tgt_path = VAL_PAIRS[0]
    inp = cv2.imread(str(inp_path), cv2.IMREAD_GRAYSCALE)
    tgt = cv2.imread(str(tgt_path), cv2.IMREAD_GRAYSCALE)
    if inp.shape != tgt.shape:
        tgt = cv2.resize(tgt, (inp.shape[1], inp.shape[0]))

    results_imgs = {'Input (degraded)': inp, 'Target (clean)': tgt}
    for name, fn in methods.items():
        out = fn(inp)
        if out.shape != inp.shape:
            out = cv2.resize(out, (inp.shape[1], inp.shape[0]))
        results_imgs[name] = out

    n = len(results_imgs)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
    fig.suptitle("Classical Methods — Visual Comparison", fontsize=12)
    for ax, (title, img) in zip(axes, results_imgs.items()):
        ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()
    print("✅ Classical comparison displayed")

except Exception as e:
    print(f"❌ Visualization error: {e}")

## Cell 6 · Data Generator

> Loads patches **one batch at a time** from disk — never loads the entire
> dataset into RAM. Augmentation is applied identically to input + target.

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import random
from pathlib import Path

class PatchDataGenerator(tf.keras.utils.Sequence):
    """Memory-safe patch generator for paired image enhancement."""

    def __init__(self, patch_dir, batch_size, patch_size, augment=False, shuffle=True):
        self.inp_dir    = Path(patch_dir) / 'input'
        self.tgt_dir    = Path(patch_dir) / 'target'
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.augment    = augment
        self.shuffle    = shuffle

        self.filenames  = sorted([f.name for f in self.inp_dir.glob('*.png')])
        if not self.filenames:
            raise FileNotFoundError(f"No patches found in {self.inp_dir}")
        self.on_epoch_end()
        print(f"✅ Generator ready: {len(self.filenames):,} patches, "
              f"{len(self):,} batches/epoch")

    def __len__(self):
        return len(self.filenames) // self.batch_size

    def __getitem__(self, idx):
        batch_fnames = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X = np.zeros((self.batch_size, self.patch_size, self.patch_size, 1), dtype=np.float32)
        Y = np.zeros((self.batch_size, self.patch_size, self.patch_size, 1), dtype=np.float32)

        for i, fname in enumerate(batch_fnames):
            inp = cv2.imread(str(self.inp_dir / fname), cv2.IMREAD_GRAYSCALE)
            tgt = cv2.imread(str(self.tgt_dir / fname), cv2.IMREAD_GRAYSCALE)
            if inp is None or tgt is None:
                continue
            inp = inp.astype(np.float32) / 255.0
            tgt = tgt.astype(np.float32) / 255.0

            if self.augment:
                inp, tgt = self._augment(inp, tgt)

            X[i, ..., 0] = inp
            Y[i, ..., 0] = tgt

        return X, Y

    def _augment(self, inp, tgt):
        """Apply identical random augmentations to input and target."""
        seed = random.randint(0, 2**31)

        # Random horizontal flip
        if random.random() < 0.5:
            inp = np.fliplr(inp)
            tgt = np.fliplr(tgt)

        # Random vertical flip
        if random.random() < 0.5:
            inp = np.flipud(inp)
            tgt = np.flipud(tgt)

        # Random 90° rotation
        if random.random() < 0.25:
            k = random.randint(1, 3)
            inp = np.rot90(inp, k)
            tgt = np.rot90(tgt, k)

        # Random brightness ±10%
        if random.random() < 0.5:
            factor = 1.0 + random.uniform(-0.1, 0.1)
            inp = np.clip(inp * factor, 0.0, 1.0)
            # Do NOT adjust target brightness (preserve ground truth)

        return inp, tgt

    def on_epoch_end(self):
        self.indices = list(range(len(self.filenames)))
        if self.shuffle:
            random.shuffle(self.indices)
        self.indices = [self.filenames[i] for i in self.indices]

print("✅ PatchDataGenerator class defined")

In [ ]:
# ── Instantiate generators ────────────────────────────────────────────────────
try:
    import gc
    train_gen = PatchDataGenerator(
        patch_dir  = Path(PATCH_DIR) / 'train',
        batch_size = BATCH_SIZE,
        patch_size = PATCH_SIZE,
        augment    = True,
        shuffle    = True
    )
    val_gen = PatchDataGenerator(
        patch_dir  = Path(PATCH_DIR) / 'val',
        batch_size = BATCH_SIZE,
        patch_size = PATCH_SIZE,
        augment    = False,
        shuffle    = False
    )
    print(f"   Train batches/epoch: {len(train_gen):,}")
    print(f"   Val   batches/epoch: {len(val_gen):,}")
    gc.collect()
except Exception as e:
    print(f"❌ Error creating generators: {e}")
    print("💡 Fix: Re-run Cell 4 (Patch Extraction) first")
    raise

## Cell 7 · Lightweight U-Net Definition

Architecture: 3-level encoder · bottleneck · 3-level decoder with skip connections.
~500K–1M parameters — right-sized for ~50,000 patches from 336 images.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Input

REG = regularizers.l2(1e-4)

def conv_block(x, filters, dropout=0.0):
    """Two Conv2D + BatchNorm + ReLU layers."""
    x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=REG)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=REG)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

def build_unet(patch_size=128, channels=1):
    """
    Lightweight 3-level U-Net for medical document enhancement.
    Input:  (patch_size, patch_size, channels) grayscale
    Output: (patch_size, patch_size, 1)  float32 sigmoid
    """
    inputs = Input(shape=(patch_size, patch_size, channels), name='input')

    # ── Encoder ──────────────────────────────────────────────────────────────
    e1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D(2)(e1)

    e2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D(2)(e2)

    e3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D(2)(e3)

    # ── Bottleneck ────────────────────────────────────────────────────────────
    b  = conv_block(p3, 256, dropout=0.2)

    # ── Decoder ──────────────────────────────────────────────────────────────
    u3 = layers.UpSampling2D(2)(b)
    u3 = layers.Concatenate()([u3, e3])
    d3 = conv_block(u3, 128, dropout=0.2)

    u2 = layers.UpSampling2D(2)(d3)
    u2 = layers.Concatenate()([u2, e2])
    d2 = conv_block(u2, 64, dropout=0.2)

    u1 = layers.UpSampling2D(2)(d2)
    u1 = layers.Concatenate()([u1, e1])
    d1 = conv_block(u1, 32)

    # ── Output — must be float32 for mixed precision numerical stability ──────
    outputs = layers.Conv2D(1, 1, padding='same', activation='linear',
                            dtype='float32', name='output')(d1)
    outputs = layers.Activation('sigmoid', dtype='float32')(outputs)

    return Model(inputs, outputs, name='MedReport_UNet')

print("✅ U-Net builder defined")

In [ ]:
# ── Build and summarize model ─────────────────────────────────────────────────
try:
    import gc, tensorflow as tf
    tf.keras.backend.clear_session()
    gc.collect()

    model = build_unet(patch_size=PATCH_SIZE, channels=IMG_CHANNELS)
    model.summary(line_length=90)

    total_params = model.count_params()
    print(f"\n✅ Model built: {total_params:,} parameters "
          f"({total_params/1e6:.2f} M)")

    if total_params > 5_000_000:
        print("⚠️  Parameter count is high — consider reducing filters")

except Exception as e:
    print(f"❌ Model build error: {e}")
    print("💡 Fix: Ensure TensorFlow ≥ 2.10 and restart runtime")
    raise

## Cell 8 · Loss Function & Compilation

Custom loss = **0.5 × L1 + 0.3 × SSIM + 0.2 × Edge (Sobel)** — optimized for medical
text documents where sharp edges and structural similarity matter most.

In [ ]:
import tensorflow as tf

def document_loss(y_true, y_pred):
    """
    Text-optimized composite loss.
    Cast to float32 explicitly for mixed-precision safety.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # L1 — sharper than MSE
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))

    # SSIM — preserves document layout structure
    ssim_loss = 1.0 - tf.reduce_mean(
        tf.image.ssim(y_true, y_pred, max_val=1.0)
    )

    # Edge loss (Sobel) — preserves text edges
    true_edges = tf.image.sobel_edges(y_true)
    pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss  = tf.reduce_mean(tf.abs(true_edges - pred_edges))

    return 0.5 * l1 + 0.3 * ssim_loss + 0.2 * edge_loss


def psnr_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

print("✅ Loss function and metrics defined")

In [ ]:
# ── Compile model ─────────────────────────────────────────────────────────────
try:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer = optimizer,
        loss      = document_loss,
        metrics   = [psnr_metric, ssim_metric]
    )
    print("✅ Model compiled")
    print("   Loss:    L1 (0.5) + SSIM (0.3) + Edge/Sobel (0.2)")
    print("   Metrics: PSNR, SSIM")

except Exception as e:
    print(f"❌ Compilation error: {e}")
    print("💡 Fix: Re-run Cell 7 to rebuild the model")
    raise

## Cell 9 · Training with Resume Support

> **Expected time:** ~30–45 minutes for 150 epochs on T4 GPU.
>
> - Best model is saved to Google Drive after each improvement.
> - **If Colab disconnects:** re-run this notebook from the top.
>   The checkpoint will be loaded automatically and training resumes.

In [ ]:
import os, gc, time
from pathlib import Path
import tensorflow as tf

checkpoint_path = Path(CHECKPOINT_DIR) / 'best_model.keras'
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ── Resume from checkpoint if it exists ───────────────────────────────────────
initial_epoch = 0
try:
    if checkpoint_path.exists():
        print(f"🔄 Checkpoint found — loading: {checkpoint_path}")
        model = tf.keras.models.load_model(
            str(checkpoint_path),
            custom_objects={'document_loss': document_loss,
                            'psnr_metric': psnr_metric,
                            'ssim_metric': ssim_metric}
        )
        print("✅ Model loaded from checkpoint — training will resume")
    else:
        print("ℹ️  No checkpoint found — starting fresh training")
except Exception as e:
    print(f"⚠️  Could not load checkpoint: {e} — starting fresh")

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath         = str(checkpoint_path),
        monitor          = 'val_loss',
        save_best_only   = True,
        save_weights_only= False,
        verbose          = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 10,
        min_lr   = 1e-6,
        verbose  = 1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = 30,
        restore_best_weights = True,
        verbose              = 1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir    = '/content/logs',
        update_freq= 'epoch'
    ),
]

print("✅ Callbacks configured")
print(f"   Checkpoint: {checkpoint_path}")
print(f"   EarlyStopping patience: 30 epochs")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
try:
    start = time.time()
    history = model.fit(
        train_gen,
        epochs          = EPOCHS,
        initial_epoch   = initial_epoch,
        validation_data = val_gen,
        callbacks       = callbacks,
        verbose         = 1
    )
    elapsed = time.time() - start
    print(f"\n✅ Training complete in {elapsed/60:.1f} min")
    print(f"   Best val_loss: {min(history.history['val_loss']):.4f}")
    gc.collect()

except Exception as e:
    print(f"❌ Training error: {e}")
    print("💡 Fixes:")
    print("   • OOM error → reduce BATCH_SIZE to 4 in Cell 2, re-run from Cell 6")
    print("   • Drive not mounted → re-run Cell 1")
    print("   • No patches found → re-run Cell 4")
    raise

## Cell 10 · Training History Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt

    h            = history.history
    epochs_ran   = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Training History", fontsize=13)

    # Loss
    axes[0].plot(epochs_ran, h['loss'],     label='Train loss')
    axes[0].plot(epochs_ran, h['val_loss'], label='Val loss')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True)

    # PSNR
    psnr_key = [k for k in h if 'psnr' in k and 'val' not in k]
    val_psnr_key = [k for k in h if 'psnr' in k and 'val' in k]
    if psnr_key and val_psnr_key:
        axes[1].plot(epochs_ran, h[psnr_key[0]],     label='Train PSNR')
        axes[1].plot(epochs_ran, h[val_psnr_key[0]], label='Val PSNR')
        axes[1].set_title('PSNR (dB)'); axes[1].set_xlabel('Epoch')
        axes[1].legend(); axes[1].grid(True)

    # SSIM
    ssim_key = [k for k in h if 'ssim' in k and 'val' not in k]
    val_ssim_key = [k for k in h if 'ssim' in k and 'val' in k]
    if ssim_key and val_ssim_key:
        axes[2].plot(epochs_ran, h[ssim_key[0]],     label='Train SSIM')
        axes[2].plot(epochs_ran, h[val_ssim_key[0]], label='Val SSIM')
        axes[2].set_title('SSIM'); axes[2].set_xlabel('Epoch')
        axes[2].legend(); axes[2].grid(True)

    plt.tight_layout(); plt.show()
    print("✅ Training history plotted")

except Exception as e:
    print(f"❌ Plot error: {e}")
    print("💡 Fix: Ensure training (Cell 9) completed successfully")

## Cell 11 · Full-Image Inference Function

Splits any full-size image into overlapping 128×128 patches, runs each through the
model, then stitches them back together by averaging overlapping regions.

In [ ]:
import numpy as np
import cv2
import tensorflow as tf

def enhance_full_image(image_path_or_array, model, patch_size=128, stride=64):
    """
    Enhance a full-size grayscale image using patch-based inference.

    Parameters
    ----------
    image_path_or_array : str, Path, or numpy ndarray (uint8 grayscale)
    model               : compiled Keras model
    patch_size          : must match training patch size
    stride              : overlap stride (smaller = more overlap, higher quality, slower)

    Returns
    -------
    enhanced : uint8 grayscale ndarray, same shape as input
    """
    # Load image
    if isinstance(image_path_or_array, np.ndarray):
        img = image_path_or_array.copy()
    else:
        img = cv2.imread(str(image_path_or_array), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Cannot read image: {image_path_or_array}")

    h, w  = img.shape
    img_f = img.astype(np.float32) / 255.0

    # Pad image so all patches are extractable
    pad_h = (patch_size - h % patch_size) % patch_size if h % stride != 0 else 0
    pad_w = (patch_size - w % patch_size) % patch_size if w % stride != 0 else 0
    padded = np.pad(img_f, ((0, pad_h), (0, pad_w)), mode='reflect')
    ph, pw = padded.shape

    output      = np.zeros((ph, pw), dtype=np.float64)
    weight_map  = np.zeros((ph, pw), dtype=np.float64)

    # Create a 2D Hann window for smooth blending at patch boundaries
    hann_1d     = np.hanning(patch_size).astype(np.float64)
    hann_2d     = np.outer(hann_1d, hann_1d)

    # Extract → predict → accumulate
    for r in range(0, ph - patch_size + 1, stride):
        for c in range(0, pw - patch_size + 1, stride):
            patch  = padded[r:r+patch_size, c:c+patch_size]
            patch_t = patch[np.newaxis, ..., np.newaxis].astype(np.float32)
            pred   = model.predict(patch_t, verbose=0)[0, ..., 0].astype(np.float64)
            output[r:r+patch_size, c:c+patch_size]     += pred * hann_2d
            weight_map[r:r+patch_size, c:c+patch_size] += hann_2d

    # Normalize by weight map (avoid division by zero)
    weight_map  = np.maximum(weight_map, 1e-8)
    output      = output / weight_map
    output      = np.clip(output, 0.0, 1.0)

    # Remove padding and convert back to uint8
    enhanced    = (output[:h, :w] * 255.0).astype(np.uint8)
    return enhanced

print("✅ enhance_full_image() defined — works on any image resolution")

## Cell 12 · Results on Validation Set

Load best model → run inference on validation images → display side-by-side comparisons.

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
try:
    import tensorflow as tf
    from pathlib import Path

    best_model_path = Path(CHECKPOINT_DIR) / 'best_model.keras'
    if best_model_path.exists():
        best_model = tf.keras.models.load_model(
            str(best_model_path),
            custom_objects={'document_loss': document_loss,
                            'psnr_metric':   psnr_metric,
                            'ssim_metric':   ssim_metric}
        )
        print(f"✅ Best model loaded from: {best_model_path}")
    else:
        print("⚠️  No saved checkpoint — using current model weights")
        best_model = model

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("💡 Fix: Ensure training (Cell 9) completed and Drive is mounted")
    best_model = model

In [ ]:
# ── Inference on validation images ───────────────────────────────────────────
try:
    import cv2, numpy as np, matplotlib.pyplot as plt, gc

    n_show       = min(10, len(VAL_PAIRS))
    dl_psnr_list = []
    dl_ssim_list = []

    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle("Results: Input (degraded) | Enhanced (model) | Target (ground truth)",
                 fontsize=12)

    for i in range(n_show):
        inp_path, tgt_path = VAL_PAIRS[i]
        inp = cv2.imread(str(inp_path), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(str(tgt_path), cv2.IMREAD_GRAYSCALE)
        if inp is None or tgt is None:
            continue
        if inp.shape != tgt.shape:
            tgt = cv2.resize(tgt, (inp.shape[1], inp.shape[0]))

        enhanced = enhance_full_image(inp, best_model, patch_size=PATCH_SIZE, stride=64)
        if enhanced.shape != tgt.shape:
            enhanced = cv2.resize(enhanced, (tgt.shape[1], tgt.shape[0]))

        psnr_val = compute_psnr(enhanced, tgt)
        ssim_val = compute_ssim(enhanced, tgt)
        dl_psnr_list.append(psnr_val)
        dl_ssim_list.append(ssim_val)

        axes[i, 0].imshow(inp,      cmap='gray'); axes[i, 0].set_title('Input',    fontsize=9)
        axes[i, 1].imshow(enhanced, cmap='gray'); axes[i, 1].set_title(
            f'Enhanced\nPSNR={psnr_val:.1f}dB  SSIM={ssim_val:.3f}', fontsize=9)
        axes[i, 2].imshow(tgt,      cmap='gray'); axes[i, 2].set_title('Target',   fontsize=9)
        for ax in axes[i]: ax.axis('off')

        # Save enhanced image
        save_path = Path(RESULTS_DIR) / f'enhanced_{inp_path.stem}.png'
        cv2.imwrite(str(save_path), enhanced)

    plt.tight_layout(); plt.show()

    avg_psnr = np.mean(dl_psnr_list)
    avg_ssim = np.mean(dl_ssim_list)
    print(f"\n📊 Deep Learning Results:")
    print(f"   Average PSNR: {avg_psnr:.2f} dB")
    print(f"   Average SSIM: {avg_ssim:.4f}")
    print(f"   Enhanced images saved to: {RESULTS_DIR}")
    gc.collect()

except Exception as e:
    print(f"❌ Inference error: {e}")
    print("💡 Fix: Ensure best_model is loaded (re-run cell above)")

## Cell 13 · Classical vs. Deep Learning Comparison

In [ ]:
try:
    import matplotlib.pyplot as plt, numpy as np

    # Deep learning results (already computed above)
    dl_avg_psnr = np.mean(dl_psnr_list) if dl_psnr_list else 0
    dl_avg_ssim = np.mean(dl_ssim_list) if dl_ssim_list else 0

    method_names  = list(classical_results.keys()) + ['U-Net (ours)']
    psnr_values   = [np.mean(v['psnr']) for v in classical_results.values()] + [dl_avg_psnr]
    ssim_values   = [np.mean(v['ssim']) for v in classical_results.values()] + [dl_avg_ssim]

    colors = ['#4C72B0'] * len(classical_results) + ['#DD8452']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Classical Methods vs. Deep Learning (U-Net)", fontsize=13)

    bars1 = ax1.bar(method_names, psnr_values, color=colors)
    ax1.set_title('PSNR (dB) — higher is better'); ax1.set_ylabel('PSNR (dB)')
    ax1.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars1, psnr_values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}', ha='center', fontsize=8)

    bars2 = ax2.bar(method_names, ssim_values, color=colors)
    ax2.set_title('SSIM — higher is better'); ax2.set_ylabel('SSIM')
    ax2.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars2, ssim_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontsize=8)

    plt.tight_layout(); plt.show()

    # Print table
    print(f"\n{'Method':<22} {'PSNR (dB)':>10} {'SSIM':>8}")
    print("-" * 42)
    for name, psnr_v, ssim_v in zip(method_names, psnr_values, ssim_values):
        marker = ' ◀ best' if psnr_v == max(psnr_values) else ''
        print(f"  {name:<20} {psnr_v:>10.2f} {ssim_v:>8.4f}{marker}")

except Exception as e:
    print(f"❌ Comparison error: {e}")
    print("💡 Fix: Ensure both classical (Cell 5) and DL (Cell 12) have run")

## Cell 14 · Enhance Your Own Image

In [ ]:
# ===== ENHANCE YOUR OWN IMAGE =====
# Change the path below to any medical report image on your Drive
test_image_path = '/content/drive/MyDrive/your_medical_report.png'

try:
    import cv2, matplotlib.pyplot as plt
    from pathlib import Path

    if not Path(test_image_path).exists():
        raise FileNotFoundError(
            f"Image not found: {test_image_path}\n"
            "💡 Change test_image_path to a valid path on your Drive"
        )

    original = cv2.imread(test_image_path, cv2.IMREAD_GRAYSCALE)
    enhanced = enhance_full_image(test_image_path, best_model,
                                  patch_size=PATCH_SIZE, stride=64)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(original, cmap='gray'); axes[0].set_title('Original (degraded)')
    axes[1].imshow(enhanced, cmap='gray'); axes[1].set_title('Enhanced')
    for ax in axes: ax.axis('off')
    plt.suptitle("Your Medical Report — Enhancement Result", fontsize=12)
    plt.tight_layout(); plt.show()

    # Save
    out_path = Path(RESULTS_DIR) / ('enhanced_' + Path(test_image_path).name)
    cv2.imwrite(str(out_path), enhanced)
    print(f"✅ Enhanced image saved to: {out_path}")

except FileNotFoundError as e:
    print(f"⚠️  {e}")
except Exception as e:
    print(f"❌ Error: {e}")

## Cell 15 · Save Results & Cleanup

In [ ]:
import csv, gc
from pathlib import Path

try:
    # ── Save metrics to CSV ─────────────────────────────────────────────────
    csv_path = Path(RESULTS_DIR) / 'metrics_summary.csv'
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

    rows = []
    for name in classical_results:
        psnr_v = float(np.mean(classical_results[name]['psnr'])) if classical_results[name]['psnr'] else 0.0
        ssim_v = float(np.mean(classical_results[name]['ssim'])) if classical_results[name]['ssim'] else 0.0
        rows.append({'method': name, 'avg_psnr': psnr_v, 'avg_ssim': ssim_v, 'type': 'classical'})

    if dl_psnr_list:
        rows.append({
            'method':   'U-Net',
            'avg_psnr': float(np.mean(dl_psnr_list)),
            'avg_ssim': float(np.mean(dl_ssim_list)),
            'type':     'deep_learning'
        })

    with open(str(csv_path), 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['method', 'avg_psnr', 'avg_ssim', 'type'])
        writer.writeheader(); writer.writerows(rows)

    print(f"✅ Metrics saved to: {csv_path}")

    # ── Free memory ─────────────────────────────────────────────────────────
    gc.collect()
    try:
        import tensorflow as tf
        tf.keras.backend.clear_session()
    except Exception:
        pass

    print("✅ Memory cleaned up")
    print("\n🎉 All done! Find your results at:")
    print(f"   Enhanced images: {RESULTS_DIR}")
    print(f"   Best model:      {CHECKPOINT_DIR}best_model.keras")
    print(f"   Metrics CSV:     {csv_path}")

except Exception as e:
    print(f"❌ Save error: {e}")
    print("💡 Fix: Ensure Google Drive is mounted and RESULTS_DIR exists")